# Entraînement de YOLO sur Colab

Ce notebook fine-tune un YOLO pré-entraîné sur le dataset de cartes, sur le GPU gratuit de Colab. Sur un MacBook M1 de 8 Go, le même entraînement prendrait plusieurs dizaines d'heures.

Il enchaîne : récupération du code depuis GitHub, régénération du dataset (identique à celui du repo grâce à la seed fixe), entraînement avec sauvegarde des poids sur Google Drive à chaque epoch, évaluation sur de vraies photos. L'évaluation sur le split test synthétique se fait ensuite en local.

## Avant de commencer

1. **Activer le GPU** : *Exécution → Modifier le type d'exécution → GPU T4*.
2. **Le code doit être poussé sur GitHub**, sur la branche indiquée dans les paramètres : le notebook la clone. Un changement local non poussé n'est pas pris en compte.
3. Lancer les cellules dans l'ordre. Garder l'onglet ouvert pendant l'entraînement : Colab coupe les sessions inactives. Le quota de GPU gratuit n'est pas garanti.

## Si la session est coupée (quota, navigateur fermé, ordinateur éteint)

Le disque de Colab est effacé, mais **les poids sont copiés sur Google Drive à chaque epoch** (étape 6). Pour reprendre : se reconnecter (avec un GPU), puis relancer toutes les cellules dans l'ordre. La génération du dataset repart de zéro (20 à 40 minutes, même résultat grâce à la seed fixe), puis l'entraînement **reprend à la dernière epoch sauvegardée**. Relancer la cellule d'entraînement alors qu'il est terminé ne fait rien.

## Paramètres

In [ ]:
from pathlib import Path

REPO = "https://github.com/renaudbmg/advanced-deep-learning-3a-cards.git"
BRANCH = "YOLOv8"     # la branche qui contient cards/detection
MODEL = "yolov8s.pt"  # yolov8n.pt : plus petit et plus rapide
EPOCHS = 50
PATIENCE = 10         # arrêt anticipé si la validation ne progresse plus pendant 10 epochs
BATCH = 32            # à baisser à 16 si "CUDA out of memory"
NAME = Path(MODEL).stem  # nom du dossier de résultats : runs/detect/<NAME>/
BACKUP = f"/content/drive/MyDrive/adl3a_runs/{NAME}_checkpoints"  # copie des poids à chaque epoch

## 1. Vérifier le GPU

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "Pas de GPU : l'entraînement tournerait sur CPU, beaucoup trop lent. "
    "Exécution → Modifier le type d'exécution → GPU T4, puis relancer."
)
print(torch.cuda.get_device_name(0))

## 2. Récupérer le code

Le dépôt est public : pas besoin de s'authentifier. Pour reprendre une version plus récente après un nouveau push, relancer le notebook sur une session neuve.

In [ ]:
!test -d repo || git clone --depth 1 --branch {BRANCH} {REPO} repo
%cd repo
!git log --oneline -1
!ls

## 3. Installer

Colab a déjà torch, numpy et pillow : seul Ultralytics manque.

In [ ]:
!pip install -q ultralytics

## 4. Brancher Google Drive

À faire **avant** l'entraînement : les poids y sont copiés à chaque epoch, pour ne rien perdre si la session s'arrête. Une fenêtre demande l'autorisation.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
!mkdir -p {BACKUP}

## 5. Régénérer le dataset

Les 52 cartes de chacun des 4 jeux sont téléchargées depuis Wikimedia, puis 24 000 images sont générées (20 000 train, 2 000 val, 2 000 test). Compter 20 à 40 minutes : la génération tourne sur le processeur (0,04 s par image sur un Mac M1).

In [ ]:
!python -m cards.dataset.assets
!python -m cards.dataset.generate_dataset

## 6. Entraîner

Le fine-tuning part des poids pré-entraînés sur COCO. Les poids sont écrits dans `runs/detect/<NAME>/weights/best.pt`, et copiés dans `BACKUP` (Drive) à chaque epoch.

`--resume` rend la cellule sûre à relancer : au premier lancement elle démarre un entraînement neuf, après une coupure elle reprend à la dernière epoch sauvegardée, et si l'entraînement est terminé elle ne fait rien.

In [ ]:
!python -m cards.detection.train --model {MODEL} --epochs {EPOCHS} --patience {PATIENCE} --batch {BATCH} --device 0 --backup-dir {BACKUP} --resume

## 7. Sauvegarder le résultat complet

Les poids sont déjà sur Drive (`BACKUP`). On y ajoute le dossier de résultats complet (courbes, paramètres), qui n'est pas copié à chaque epoch.

In [ ]:
!mkdir -p /content/drive/MyDrive/adl3a_runs
!cp -r runs/detect/{NAME} /content/drive/MyDrive/adl3a_runs/
!ls /content/drive/MyDrive/adl3a_runs/{NAME}

En secours, ou pour récupérer directement le fichier, télécharger les poids :

In [ ]:
from google.colab import files

files.download(f"runs/detect/{NAME}/weights/best.pt")

## 8. Évaluer sur de vraies photos (facultatif)

Le test qui compte : de vraies photos, jamais vues à l'entraînement. Le split `validation` sert à comparer les réglages, le split `test` donne le chiffre final. Quelques minutes.

In [ ]:
!python -m cards.detection.real_dataset --split validation
!python -m cards.detection.real_dataset --split test
!python -m cards.detection.evaluate_real --weights runs/detect/{NAME}/weights/best.pt --data-dir data/real/poker-cards/validation
!python -m cards.detection.evaluate_real --weights runs/detect/{NAME}/weights/best.pt
!cp -r runs/real_eval /content/drive/MyDrive/adl3a_runs/{NAME}/

## Suite, en local

1. Récupérer `best.pt` (téléchargement ci-dessus, ou depuis `MyDrive/adl3a_runs/<NAME>_checkpoints/best.pt`).
2. Évaluer sur le split test, depuis la racine du repo :
   ```bash
   python -m cards.detection.evaluate --weights best.pt --map
   ```
3. Reporter les chiffres dans le tableau « Résultats » du README de la détection.

Pour choisir le seuil de confiance `--conf`, lancer l'évaluation sur le split **val** (`--split-dir data/synthetic/val`), pas sur le test.